In [ ]:
from google.colab import drive, files
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


匯入
載入 Keras load_model，以及 numpy/pandas/opencv/tqdm 等

路徑與常數

指定 CSV_PATH 只用來推斷訓練波段數（BANDS），MODEL_PATH 指向訓練出的 .keras 模型；定義四類標籤與對應 BGR 顏色（BG=灰、PET=藍、PP=紅、PS=綠）。

In [ ]:
import os, glob, time
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from tensorflow.keras.models import load_model

# ===== 路徑 =====
CSV_PATH   = "/content/drive/MyDrive/BL0909/M/ALL.csv"   # 用來推斷波段數(=影像高度)
MODEL_PATH = "/content/drive/MyDrive/BL0909/M/MODEL/Conv1D_BL20260331.keras"
OUT_DIR   = "/content/drive/MyDrive/BL0909/M/MODEL"
IMAGE_DIR = "/content/drive/MyDrive/BL99"
os.makedirs(OUT_DIR, exist_ok=True)

# ===== 類別順序（一定要和訓練時一致）=====
LABELS = ["BG", "PET", "PP", "PS"]

# ===== 顏色（OpenCV: BGR）=====
COLORS_IDX = {
    0: (50, 50, 50),   # BG 深灰
    1: (255, 0, 0),    # PET 藍
    2: (0, 0, 255),    # PP 紅
    3: (0, 255, 0),    # PS 綠
}

# ===== 由 CSV 推斷波段數 =====
df = pd.read_csv(CSV_PATH)
BANDS = df.shape[1] - 1   # 若最後一欄是 label
print("BANDS =", BANDS)

# ===== 載入模型 =====
model = load_model(MODEL_PATH)
print("Model loaded successfully.")

BANDS = 1232
Model loaded successfully.


In [ ]:
import pandas as pd

# ===== 從 CSV 推斷波段數 =====
df_head = pd.read_csv(CSV_PATH, nrows=1)
feat_cols = df_head.columns[:-1]   # 最後一欄是 label
BANDS = len(feat_cols)

print("BANDS =", BANDS)

BANDS = 1232


載入模型 & 確認輸入維度

load_model 載入後，讀出模型期望輸入維度（應為 1235，用於後續一致性檢查）。

從 CSV 推斷 BANDS

讀 ALL.csv 的前 1 列，數一下最後一欄以外的欄位數 → 得到訓練時的波段數（例如 1232）

In [ ]:
# 載入模型
print("Loading model...")
model = load_model(MODEL_PATH, compile=False)
print("Model loaded.")

# 模型輸入列表：預期 [seq_input(?, 1232, 1), b3_input(?, 3)]
for i, t in enumerate(model.inputs):
    print(f"Input {i} shape:", t.shape)

# 從 CSV 抓欄位數來推斷波段數 BANDS（= 影像高度）
print("Inferring bands from CSV header...")
df_head = pd.read_csv(CSV_PATH, nrows=1)
feat_cols = df_head.columns[:-1]     # 除最後一欄 Label
BANDS = len(feat_cols)               # 常見 1232
print("BANDS =", BANDS)


Loading model...
Model loaded.
Input 0 shape: (None, 1232, 1)
Inferring bands from CSV header...
BANDS = 1232


In [ ]:
import pandas as pd
import numpy as np

# ===== 讀 CSV（你漏掉的）=====
df = pd.read_csv(CSV_PATH)

print("資料讀取完成")
print("欄位：", df.columns.tolist())

# ===== 特徵與標籤 =====
X = df.iloc[:, :-1].to_numpy(dtype=np.float32)
y = df.iloc[:, -1].astype(str).str.upper().str.strip()

print("類別：", np.unique(y))

# ===== reshape 給 CNN =====
X_seq = X.reshape((X.shape[0], X.shape[1], 1))

print("CNN輸入:", X_seq.shape)
print("樣本數:", X.shape[0])
print("波段數:", X.shape[1])


資料讀取完成
欄位： ['366.217137', '366.7177379535301', '367.2184383483211', '367.71923808567374', '368.2201370668889', '368.72113519326746', '369.22223236611023', '369.723428486718', '370.2247234563917', '370.726117176432', '371.22760954814', '371.72920047281633', '372.2308898517619', '372.73267758627753', '373.2345635776641', '373.7365477272225', '374.2386299362534', '374.7408101060578', '375.24308813793647', '375.7454639331903', '376.24793739312', '376.7505084190265', '377.2531769122107', '377.75594277397335', '378.25880590561536', '378.7617662084375', '379.2648235837406', '379.7679779328256', '380.2712291569933', '380.77457715754446', '381.27802183578', '381.7815630930007', '382.2852008305075', '382.7889349496012', '383.29276535158255', '383.7966919377525', '384.30071460941184', '384.8048332678614', '385.3090478144021', '385.8133581503346', '386.31776417696', '386.8222657955789', '387.3268629074923', '387.83155541400095', '388.33634321640574', '388.8412262160075', '389.346204314107', '389.8

In [ ]:
def build_dual_inputs_from_gray(gray_u8: np.ndarray, do_minmax_per_col: bool=False):
    """
    gray_u8: (H, W)，H 必須等於 BANDS。
    回傳：
      X_seq : (W, BANDS, 1)  → Conv1D 分支
      X_b3  : (W, 3)         → B+3 分支
    """
    H, W = gray_u8.shape
    assert H == BANDS, f"H={H} 與 BANDS={BANDS} 不符"

    # 轉成 (W, BANDS)
    X_raw = gray_u8.T.astype(np.float32)

    if do_minmax_per_col:
        # 依每一欄做 0~1（若訓練時有做相同前處理，推論端要一致）
        min_v = X_raw.min(axis=1, keepdims=True)
        max_v = X_raw.max(axis=1, keepdims=True)
        denom = np.maximum(max_v - min_v, 1e-6)
        X_raw = (X_raw - min_v) / denom

    # Conv1D 分支需求：(W, BANDS, 1)
    X_seq = X_raw[:, :, np.newaxis]

    # B+3 分支：(W, 3)
    X_b3_list = [spectrum_to_Bplus3(row) for row in X_raw]
    X_b3 = np.stack(X_b3_list, axis=0)

    return X_seq, X_b3


視覺化工具：mixcolor_by_confidence

* 將每一欄的預測類別上色（依類別 BGR），亮度乘上該欄最大機率（置信度）→ 產生 (1, W, 3) 彩色條帶。






In [ ]:
def colorize_row_by_conf(pred_idx: np.ndarray, conf: np.ndarray) -> np.ndarray:
    """
    pred_idx: (W,)  conf: (W,)
    產出 (1, W, 3) BGR 彩色列，置信度越高越亮。
    """
    rows = []
    for c, p in zip(conf, pred_idx):
        base = np.array(COLORS[int(p)], dtype=np.float32)  # (3,)
        rows.append((base * float(c)).clip(0, 255))
    row_bgr = np.vstack(rows).astype(np.uint8)             # (W, 3)
    return row_bgr[np.newaxis, :, :]                       # (1, W, 3)


In [ ]:
def build_cnn_input_from_gray(gray_u8, do_minmax_per_col=False):
    """
    gray_u8: (H, W)，H 必須等於 BANDS
    回傳: (W, BANDS, 1)
    """
    H, W = gray_u8.shape
    assert H == BANDS, f"H={H} 與 BANDS={BANDS} 不符"

    # (W, BANDS)
    X_raw = gray_u8.T.astype(np.float32)

    if do_minmax_per_col:
        min_v = X_raw.min(axis=1, keepdims=True)
        max_v = X_raw.max(axis=1, keepdims=True)
        denom = np.maximum(max_v - min_v, 1e-6)
        X_raw = (X_raw - min_v) / denom

    # (W, BANDS, 1)
    X_seq = X_raw[:, :, np.newaxis]

    return X_seq

In [ ]:
def build_cnn_input_from_gray(gray, do_minmax=False):
    # gray shape: (BANDS, W)
    X = gray.T.astype(np.float32)   # -> (W, BANDS)

    if do_minmax:
        mn = X.min(axis=1, keepdims=True)
        mx = X.max(axis=1, keepdims=True)
        X = (X - mn) / (mx - mn + 1e-8)

    X = X[..., np.newaxis]          # -> (W, BANDS, 1)
    return X

In [ ]:
def colorize_row_by_conf(pred_idx, conf):
    rows = []
    for c, p in zip(conf, pred_idx):
        base = np.array(COLORS_IDX[int(p)], dtype=np.float32)
        rows.append((base * float(c)).clip(0, 255))

    row_bgr = np.vstack(rows).astype(np.uint8)   # (W, 3)
    return row_bgr[np.newaxis, :, :]             # (1, W, 3)

In [ ]:
img_paths = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.jpg")))
print(f"Found {len(img_paths)} JPG in {IMAGE_DIR}")

tiles = []
per_img_stats = []
start = time.time()

# 是否要每條光譜做 min-max（需和訓練前處理一致）
DO_MINMAX_PER_COLUMN = False

for p in tqdm(img_paths, desc="Predicting"):
    gray = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if gray is None:
        print("跳過（讀取失敗）：", p)
        continue

    H, W = gray.shape
    if H != BANDS:
        raise ValueError(f"高度不符：{os.path.basename(p)} 高度={H}, 需要={BANDS}")

    # 構建 CNN 單輸入
    X_seq = build_cnn_input_from_gray(gray, DO_MINMAX_PER_COLUMN)   # (W, BANDS, 1)

    # 預測
    probs = model.predict(X_seq, verbose=0)   # (W, 4)
    pred  = np.argmax(probs, axis=1)          # (W,)
    conf  = np.max(probs, axis=1)             # (W,)

    # 統計
    uniq, cnt = np.unique(pred, return_counts=True)
    stats = {"file": os.path.basename(p), "mean_conf": float(conf.mean())}
    for k, v in zip(uniq, cnt):
        stats[LABELS[int(k)]] = int(v)
    per_img_stats.append(stats)

    # 做一條彩色列
    tile = colorize_row_by_conf(pred, conf)   # (1, W, 3)
    tiles.append(tile)

elapsed = time.time() - start
print(f"Done. Time: {elapsed:.2f}s")

Found 500 JPG in /content/drive/MyDrive/BL99


Predicting: 100%|██████████| 500/500 [12:38<00:00,  1.52s/it]

Done. Time: 758.39s


In [ ]:
if tiles:
    recon = np.vstack(tiles)  # (N, W, 3)
    out_img = os.path.join(OUT_DIR, "Conv1D__0331BlStats.jpg")
    cv2.imwrite(out_img, recon)
    print("✅ Saved image:", out_img)

    # 每張圖的像素統計
    df_stats = pd.DataFrame(per_img_stats).fillna(0)
    # 確保四類欄位齊全
    for lab in LABELS:
        if lab not in df_stats.columns:
            df_stats[lab] = 0
    df_stats = df_stats[["file", "mean_conf"] + LABELS]
    out_csv = os.path.join(OUT_DIR, "Conv1D__0331BlStats.csv")
    df_stats.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print("✅ Saved stats:", out_csv)
else:
    print("⚠️ 沒有處理到任何圖片。")

✅ Saved image: /content/drive/MyDrive/BL0909/M/MODEL/Conv1D__0331BlStats.jpg
✅ Saved stats: /content/drive/MyDrive/BL0909/M/MODEL/Conv1D__0331BlStats.csv


輸出結果

* 若有處理到影像：
vstack(tiles) 變成整張重建圖（N×W×3），輸出 MLP_Bplus3_Recon_YYYYMMDD.jpg。

* 建立統計 DataFrame，補齊四類欄，再輸出 MLP_Bplus3_PerImage_Stats.csv（含 file、mean_conf、BG/PET/PP/PS 欄）。


In [ ]:
# 嘗試讀取剛剛輸出的重建圖檔
latest_imgs = sorted(glob.glob(os.path.join(OUT_DIR, "_Bplus3_Recon_*.jpg")))
if latest_imgs:
    recon_path = latest_imgs[-1]
    recon_bgr  = cv2.imread(recon_path, cv2.IMREAD_COLOR)
    recon_rgb  = cv2.cvtColor(recon_bgr, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12, 12))
    plt.imshow(recon_rgb)
    plt.axis('off')
    plt.title(f"Reconstructed Classification (B+3) - {os.path.basename(recon_path)}")
    plt.show()
else:
    print("❌ 找不到重建圖檔，請確認前面推論有正常輸出。")


❌ 找不到重建圖檔，請確認前面推論有正常輸出。


In [ ]:
# === 補：色盤 & 上色工具 ===
import numpy as np

def build_index_to_color(labels):
    """
    依照 labels 的順序回傳 {index: BGR} 對應。
    與 RF/MLP 規則一致：BG=白、PET=藍、PS=綠、PP=紅（OpenCV 用 BGR）。
    若有未知 label 就給中性灰。
    """
    base = {
        "BG":  (255, 255, 255),  # 白
        "PET": (255,   0,   0),  # 藍 (BGR)
        "PS":  (  0, 255,   0),  # 綠
        "PP":  (  0,   0, 255),  # 紅
    }
    return {
        i: np.array(base.get(name, (160, 160, 160)), dtype=np.uint8)
        for i, name in enumerate(labels)
    }

def color_row_from_pred(pred, index_to_color, use_confidence_shading=False, probs=None, row_height=4):
    """
    將一列的類別索引 pred (W,) 轉成彩色條 (row_height, W, 3)。
    use_confidence_shading=True 時，會用 conf 亮度加權（需提供 probs）。
    """
    pred = np.asarray(pred).astype(int)
    W = pred.shape[0]
    row = np.zeros((row_height, W, 3), dtype=np.uint8)

    # 依 pred 取色
    colors = np.stack([index_to_color.get(int(i), np.array([160,160,160], np.uint8)) for i in pred], axis=0)  # (W,3)
    row[:] = colors[None, :, :]

    # 依信心值調整亮度（可選）
    if use_confidence_shading and probs is not None:
        conf = probs.max(axis=1).astype(np.float32)  # (W,)
        scale = (0.6 + 0.4 * conf).astype(np.float32)  # 亮度 0.6~1.0
        row = (row.astype(np.float32) * scale[None, :, None]).clip(0, 255).astype(np.uint8)

    return row
